# Sentiment Analysis Studio

**Goal:** classify short English reviews as positive (`1`) or negative (`0`) using the three original Qiyas text files.

This notebook is the reproducible modeling walkthrough. The Streamlit app in the project root uses the model saved by this workflow.

## Project workflow
1. Load and inspect the three source files.
2. Clean and combine the data.
3. Create a stratified train/test split.
4. Convert text to TF-IDF features.
5. Compare Logistic Regression, Linear SVM, and Multinomial Naive Bayes.
6. Evaluate the best model and save it for the app.

> Important: this is a binary sentiment dataset. It does not contain neutral labels, and the model should not be assumed to work equally well on other languages or domains.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, ConfusionMatrixDisplay
)

ROOT = Path.cwd()
# If running this notebook from notebooks/, use the project root instead:
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
print("Project root:", ROOT.resolve())

## 2. Load the three datasets
Each row contains review text followed by a tab and its sentiment label.

In [ ]:
files = ["one.txt", "two.txt", "three.txt"]
frames = []

for filename in files:
    frame = pd.read_csv(
        RAW_DIR / filename,
        sep="\\t",
        names=["text", "label"],
        quoting=3,
        encoding="utf-8",
        on_bad_lines="skip"
    )
    frame["source"] = Path(filename).stem
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)
print("Combined shape:", df.shape)
display(df.head())

## 3. Inspect and clean the data

In [ ]:
df["text"] = df["text"].fillna("").astype(str).str.strip()
df = df[(df["text"] != "") & df["label"].isin([0, 1])]
df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("\\nLabel counts:")
display(df["label"].value_counts().rename(index={0: "Negative", 1: "Positive"}))
print("\\nRows by source:")
display(df["source"].value_counts())

In [ ]:
ax = df["label"].map({0: "Negative", 1: "Positive"}).value_counts().plot(
    kind="bar", title="Sentiment label distribution", rot=0, figsize=(6, 4)
)
ax.set_ylabel("Number of examples")
plt.tight_layout()
plt.show()

## 4. Split the data
We keep 20% of the data aside for final evaluation. `stratify` keeps the class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Build and compare models
TF-IDF gives more weight to terms that are informative within a document collection. Unigrams are single words; bigrams are two-word phrases.

In [ ]:
def make_vectorizer():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
        strip_accents="unicode"
    )

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", make_vectorizer()),
        ("classifier", LogisticRegression(max_iter=1500, C=2.0))
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", make_vectorizer()),
        ("classifier", LinearSVC(C=1.0))
    ]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", make_vectorizer()),
        ("classifier", MultinomialNB(alpha=0.5))
    ])
}

results = []
fitted_models = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions, average="binary", zero_division=0
    )
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })
    fitted_models[name] = pipeline

comparison = pd.DataFrame(results).sort_values("f1", ascending=False)
display(comparison)

## 6. Evaluate the selected model
The model with the highest F1-score on the test set is selected. F1 balances precision and recall.

In [ ]:
best_name = comparison.iloc[0]["model"]
best_model = fitted_models[best_name]
test_predictions = best_model.predict(X_test)

print("Selected model:", best_name)
print(classification_report(
    y_test, test_predictions,
    target_names=["Negative", "Positive"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions,
    display_labels=["Negative", "Positive"],
    cmap="Blues"
)
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

## 7. Save the model and evaluation results
The saved pipeline includes both TF-IDF vectorization and the classifier, so the app can accept raw text directly.

In [ ]:
joblib.dump(best_model, MODEL_DIR / "sentiment_pipeline.joblib")
comparison.to_csv(REPORT_DIR / "model_comparison.csv", index=False)

report = classification_report(
    y_test, test_predictions,
    target_names=["Negative", "Positive"],
    output_dict=True,
    zero_division=0
)
metrics = {
    "best_model": best_name,
    "dataset": {
        "rows": int(len(df)),
        "positive": int((df.label == 1).sum()),
        "negative": int((df.label == 0).sum()),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test))
    },
    "best_model_metrics": {
        "accuracy": float(accuracy_score(y_test, test_predictions)),
        "precision": float(report["Positive"]["precision"]),
        "recall": float(report["Positive"]["recall"]),
        "f1": float(report["Positive"]["f1-score"])
    },
    "classification_report": report,
    "model_comparison": comparison.to_dict("records"),
    "label_mapping": {"0": "Negative", "1": "Positive"}
}
(REPORT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved model and reports.")

## 8. Try a new sentence

In [ ]:
examples = [
    "The food was delicious and the staff were friendly.",
    "The product broke quickly and customer support was terrible."
]
for sentence in examples:
    label = int(best_model.predict([sentence])[0])
    sentiment = "Positive" if label == 1 else "Negative"
    print(f"{sentiment}: {sentence}")

## 9. Interpretation and limitations

- The test set is separate from the training data, so the reported test metrics estimate performance on unseen examples drawn from a similar distribution.
- The dataset is balanced overall, but it combines different review domains (for example, products, restaurants, and movies).
- TF-IDF mostly learns from word and phrase patterns. It does not truly understand context.
- Sarcasm, negation in complex sentences, spelling variation, and domain changes can cause mistakes.
- The output is binary: positive or negative. There is no neutral class.

**Next steps:** test with your own examples, inspect false positives and false negatives, and consider a larger dataset or contextual language model if you need stronger generalization.